# Router **auto-detectado** (idea de ERMoE): ¿el parche sabe solo cuándo encenderse? — Qwen3.5-0.8B (Colab L4)

**ERMoE** (arXiv 2511.10971, CVPR 2026) no entrena un router aparte: manda cada token al experto cuya "base" se le
parece más (similitud coseno). Aquí el "experto" es el parche de loops **p50c**: su dirección `u` vive en la salida de
las capas que edita (L15 y L19). **Self-gate** = qué tanto apunta cada token hacia `u`. Si los tokens del loop apuntan
hacia `u`, el parche se puede encender solo, sin router entrenado.

Se compara, en respuestas apartadas (mitad B; la mitad A solo fija umbrales):
- **self-gate**: la `u` del propio parche (coseno y producto punto)
- **probe**: un router aprendido (diferencia de medias por capa, ajustado en la mitad A) — lo que se usa hoy en otros parches
- **random**: una dirección al azar en las mismas capas (control)
- **router de texto**: el detector actual (50 % de los últimos 64 tokens repetidos)

Métricas: AUC por token (loop vs normal) y alarma temprana con 1 % de falsas alarmas: ¿cuántos loops detecta y
**cuántos tokens antes** que el detector de texto?

Reusa las respuestas del base ya guardadas en Drive (GSM8K `discover_blind` y MATH-500 `discover_math500_qwen35-08b`).
**Veto (v2):** el detector de texto propone y la dirección del parche confirma: ¿cuántas falsas alarmas quita y cuánto
retrasa el encendido? Se mide a 1 / 5 / 10 / 20 % de falsas alarmas por token.

No genera nada: solo pasa esas respuestas por el modelo. ~5–10 min en L4.
Resultado: `MyDrive/wsi_runs/FINALES/self_gate_veto_qwen35-08b.json`. Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "self_gate_veto_qwen35-08b"
RUNS = {"gsm8k": "discover_blind", "math": "discover_math500_qwen35-08b"}   # carpetas en MyDrive/wsi_runs
REPO = "/content/wsi"
BASE = "/content/base"
S = f"{REPO}/scripts/experiment_self_gate_router.py"
PATCH = f"{REPO}/patches/qwen35_0.8b_p50c_loop_fix.moipatch"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q hf_transfer
!pip install -q flash-linear-attention causal-conv1d || echo "fla opcional"
import torch, transformers; print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, datetime, time
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/wsi_runs"
FINALES = f"{ROOT}/FINALES"
OUTD = f"{ROOT}/{RUN_NAME}"
os.makedirs(FINALES, exist_ok=True); os.makedirs(OUTD, exist_ok=True)
WORK = {}
for ds, run in RUNS.items():
    src = f"{ROOT}/{run}"
    need = ("config.json", "requests_test.jsonl", "outputs_test_base.jsonl")
    if not all(os.path.exists(f"{src}/{f}") for f in need):
        print(f"{ds}: faltan archivos en {src} -> se salta")
        continue
    w = f"/content/work_{ds}"
    os.makedirs(w, exist_ok=True)
    for f in need:
        shutil.copy2(f"{src}/{f}", w)
    WORK[ds] = w
    print(ds, "<-", src)
assert WORK, "no hay ninguna corrida que reusar en Drive"

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert os.path.exists(S), "el repo no tiene experiment_self_gate_router.py: falta push"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
RES = {}
for ds, w in WORK.items():
    out = f"{OUTD}/{ds}.json"
    if not os.path.exists(out):                       # retoma: lo ya hecho no se repite
        print(f"\n===== {ds} =====", flush=True)
        !cd {REPO}/scripts && python -u {S} --work {w} --model {BASE} --patch {PATCH} --out /content/sg_{ds}.json
        shutil.copy2(f"/content/sg_{ds}.json", out)
    RES[ds] = json.load(open(out))

In [ ]:
for ds, r in RES.items():
    t = r["text_router"]
    print(f"\n{ds}: {r['n_half_b']} respuestas apartadas, {r['loop_rate_b']:.0%} con loop")
    print(f"  router de texto  : detecta {t['loops_caught']:.0%} | se dispara {t['median_fire_after_onset']:.0f} tokens después del inicio")
    for name, v in [("self-gate (mejor)", r["self_gate_best"]), (f"probe L{r['probe_best']['layer']} (mejor)", r["probe_best"])] + \
                   [(k.replace("random_", "azar ").replace("_cos", ""), v) for k, v in r["random"].items() if k.endswith("_cos")]:
        print(f"  {name:32s} AUC {v['auc']:.3f} | detecta {v['loops_caught']} | adelanto vs texto {v['median_lead_vs_text_router']} tok"
              f" | normales que dispara {v['normal_answers_fired']}")
    print("  veto (texto propone, el parche confirma):")
    for name in ("all_modules_cos", *[k for k in r["veto"] if k.startswith("self_")]):
        for fpr, m in r["veto"][name].items():
            print(f"    {name:36s} {float(fpr):4.0%} | loops encendidos {m['loops_caught']:.0%} | normales encendidas "
                  f"{m['normal_answers_on']} (solo texto {t['normal_answers_fired']}) | retraso {m['median_delay_after_text']} tok")
out = {"runs": RES}
out["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "self_gate_router_colab.ipynb",
                 "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as fh:
    json.dump(out, fh, indent=1)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")